# Master pipeline v2 — per-pair-best-distribution modelling (data3)

Revised order (VIF inserted early; the old two-family covariate screen collapsed into one
that uses **each pair's own best distribution**):

1. Inferential statistics — Overall / BTW / PR p-values side by side, symbols, magnitude → **Excel 1**
2. **VIF** multicollinearity check across covariates (incl. Site) → **Excel 2**
3. Distribution fitting per pair (all candidates, report all) → **Excel 3**
4. Drop low-sample pairs (editable)
5. Covariate screening per kept pair **under that pair's own best distribution** (incl. Site) → **Excel 4**
6. Finalize covariate set — VIF-driven de-correlation (speed & speed-difference kept together
   when not collinear; flow/site collapsed when collinear) → **Excel 5**

AFT models fix `loc = 0` (nested covariate LR tests); marginal fitting (Step 3) uses free `loc`.
Note: covariate significance is distribution-dependent — pairs whose best fit is `genextreme`
are flagged in Step 5.

In [1]:
# --- Cell 1: Imports, paths, configuration ---
import os, warnings
import numpy as np
import pandas as pd
from scipy import stats, optimize
warnings.simplefilter("ignore")

BASE      = r"D:\Headway"
DATA_PATH = os.path.join(BASE, "data3.xlsx")
TABLES    = os.path.join(BASE, "Tables")
os.makedirs(TABLES, exist_ok=True)

OUTCOME       = "Time_Headway"
ALPHA         = 0.05
DAIC_MIN      = 2.0     # strict-improvement AIC threshold
MIN_BIN_GROUP = 5
MIN_N_FIT     = 30
VIF_THRESH    = 5.0     # max acceptable VIF when adding a covariate at finalize

CANDIDATE_DISTS = ["lognorm", "gamma", "weibull_min", "invgauss", "expon",
                   "fisk", "pearson3", "gengamma", "genextreme", "rayleigh"]
# distributions that are extreme-value (flag for scrutiny, not standard headway models)
FLAG_DISTS = {"genextreme"}

SYMBOL = {"Spearman": "\u03c1", "Mann-Whitney U": "r", "Kruskal-Wallis": "\u03b5\u00b2"}

def magnitude(es, symbol):
    a = abs(es)
    if symbol in ("\u03c1", "r"):                 # correlation-type
        return ("negligible" if a < 0.10 else "small" if a < 0.30 else
                "medium" if a < 0.50 else "large")
    else:                                          # epsilon-squared
        return ("negligible" if a < 0.01 else "small" if a < 0.06 else
                "medium" if a < 0.14 else "large")

def fmt_p(p):
    return "<0.001" if (pd.notna(p) and p < 1e-3) else ("n/a" if pd.isna(p) else round(float(p), 4))

In [2]:
# --- Cell 2: Load data3, audit, build covariate registry (incl. Site) ---
df = pd.read_excel(DATA_PATH)
SUBJECT_COL = "V_Target" if "V_Target" in df.columns else "V_Subject"
SPEED_COL   = "Target_Speed_km/hr" if "Target_Speed_km/hr" in df.columns else "Subject_Speed_km/hr"
BUS_COL     = next((c for c in df.columns if "bus" in c.lower()), None)

# numeric encodings for binary covariates
df["_site01"] = (df["Site"] == "Shahjahanpur").astype(int)
df["_offc01"] = df["Off_centeredness"].astype(int)
df["_occ01"]  = df["Occupancy"].astype(int)

print(f"rows={len(df)}  subject={SUBJECT_COL}  speed={SPEED_COL}  share_bus={BUS_COL}  missing={int(df.isna().sum().sum())}")
print("\nPair sizes:")
for p, n in df["Pair"].value_counts().sort_values(ascending=False).items():
    print(f"  {p:24s} n={n:4d}  [{'ok' if n>=MIN_N_FIT else 'small' if n>=15 else 'TOO SMALL'}]")

REGISTRY = {
    "speed":      dict(col=SPEED_COL,          type="cont", unit=1,    label="per +1 km/h"),
    "speed_diff": dict(col="Speed_Difference", type="cont", unit=1,    label="per +1 km/h"),
    "flow":       dict(col="Flow_pcu/hr",      type="cont", unit=1000, label="per +1000 pcu/hr"),
    "off_cen":    dict(col="_offc01",          type="bin",  unit=1,    label="True vs False"),
    "occupancy":  dict(col="_occ01",           type="bin",  unit=1,    label="True vs False"),
    "site":       dict(col="_site01",          type="bin",  unit=1,    label="Shahjahanpur vs Tikatuli"),
}
if BUS_COL is not None:
    s = pd.to_numeric(df[BUS_COL], errors="coerce")
    REGISTRY["share_bus"] = (dict(col=BUS_COL, type="cont", unit=0.1,  label="per +0.10 share")
                             if s.max() <= 1.5 else
                             dict(col=BUS_COL, type="cont", unit=10.0, label="per +10 (pct pts)"))
COVARIATES = {k: v for k, v in REGISTRY.items() if v["col"] in df.columns}
print("\nCovariates:", list(COVARIATES.keys()))

rows=898  subject=V_Target  speed=Target_Speed_km/hr  share_bus=None  missing=0

Pair sizes:
  BTW_following_4W         n= 250  [ok]
  BTW_following_MT_3W      n= 186  [ok]
  BTW_following_NMT_3W     n= 119  [ok]
  PR_following_MT_3W       n= 104  [ok]
  BTW_following_MT_2W      n=  73  [ok]
  PR_following_NMT_3W      n=  49  [ok]
  PR_following_4W          n=  43  [ok]
  BTW_following_NMT_2W     n=  41  [ok]
  PR_following_MT_2W       n=  23  [small]
  PR_following_NMT_2W      n=  10  [TOO SMALL]

Covariates: ['speed', 'speed_diff', 'flow', 'off_cen', 'occupancy', 'site']


In [3]:
# --- Cell 3: Association helper (symbols + magnitude) ---
def rank_biserial(u, n1, n2): return 1 - (2*u)/(n1*n2)
def epsilon_squared(h, n):    return h/(n-1)
def bh_adjust(pvals):
    p = np.asarray(pvals, float); n = len(p); order = np.argsort(p)
    ranked = np.minimum.accumulate((p[order]*n/np.arange(1, n+1))[::-1])[::-1]
    adj = np.empty(n); adj[order] = np.clip(ranked, 0, 1); return adj

def associate(data, variables):
    """Return dict: var -> (Scale, Test, Symbol, ES, p_raw)."""
    recs = {}
    for var, scale in variables:
        sub = data[[var, OUTCOME]].dropna(); n = len(sub)
        if scale == "cont":
            if sub[var].std() == 0: continue
            es, p = stats.spearmanr(sub[var], sub[OUTCOME]); test = "Spearman"
        elif scale == "bin":
            grp = [g[OUTCOME].values for _, g in sub.groupby(var)]
            if len(grp) != 2 or min(map(len, grp)) < 3: continue
            u, p = stats.mannwhitneyu(grp[0], grp[1], alternative="two-sided")
            es = rank_biserial(u, len(grp[0]), len(grp[1])); test = "Mann-Whitney U"
        else:
            grp = [g[OUTCOME].values for _, g in sub.groupby(var)]
            if len(grp) < 2: continue
            h, p = stats.kruskal(*grp); es = epsilon_squared(h, n); test = "Kruskal-Wallis"
        recs[var] = (scale, test, SYMBOL[test], round(es, 4), p)
    # BH within this group
    if recs:
        ps = bh_adjust([v[4] for v in recs.values()])
        for (k, v), padj in zip(list(recs.items()), ps):
            recs[k] = v + (padj,)     # append p_BH
    return recs

In [4]:
# --- Cell 4: STEP 1 - Inferential statistics (Overall/BTW/PR side by side) -> Excel 1 ---
base_vars = [("V_Leading_Class","multi"), ("Pair","multi"),
             (SPEED_COL,"cont"), ("Leading_Speed_km/hr","cont"), ("Speed_Difference","cont"),
             ("Off_centeredness","bin"), ("Occupancy","bin"), ("Flow_pcu/hr","cont"), ("Site","bin")]
overall_vars = [(SUBJECT_COL,"bin")] + base_vars

rec_all = associate(df, overall_vars)
rec_btw = associate(df[df[SUBJECT_COL]=="BTW"], base_vars)
rec_pr  = associate(df[df[SUBJECT_COL]=="PR"],  base_vars)

def cell(rec, var, which):
    if var not in rec: return (np.nan, "", np.nan)
    scale, test, sym, es, p_raw, p_bh = rec[var]
    return (es, magnitude(es, sym), p_bh)

rows = []
for var, _ in overall_vars:
    scale, test, sym = (rec_all[var][0], rec_all[var][1], rec_all[var][2]) if var in rec_all else ("", "", "")
    esO, magO, pO = cell(rec_all, var, "O")
    esB, magB, pB = cell(rec_btw, var, "B")
    esP, magP, pP = cell(rec_pr,  var, "P")
    rows.append(dict(Variable=var, Scale=scale, Test=test, Symbol=sym,
                     ES_Overall=esO, Mag_Overall=magO, p_Overall=fmt_p(pO),
                     ES_BTW=esB, Mag_BTW=magB, p_BTW=fmt_p(pB),
                     ES_PR=esP, Mag_PR=magP, p_PR=fmt_p(pP)))
inferential = pd.DataFrame(rows)

# tidy per-group detail sheets too
def detail(rec):
    out = [dict(Variable=k, Scale=v[0], Test=v[1], Symbol=v[2], EffectSize=v[3],
                Magnitude=magnitude(v[3], v[2]), p_raw=fmt_p(v[4]), p_BH=fmt_p(v[5]),
                Significant="Yes" if (pd.notna(v[5]) and v[5]<ALPHA) else "No") for k, v in rec.items()]
    d = pd.DataFrame(out)
    return d.reindex(d["EffectSize"].abs().sort_values(ascending=False).index).reset_index(drop=True) if len(d) else d

xl1 = os.path.join(TABLES, "01_inferential_stats.xlsx")
with pd.ExcelWriter(xl1, engine="openpyxl") as xl:
    inferential.to_excel(xl,  sheet_name="Summary_PR_BTW", index=False)
    detail(rec_all).to_excel(xl, sheet_name="Overall_detail", index=False)
    detail(rec_btw).to_excel(xl, sheet_name="BTW_detail", index=False)
    detail(rec_pr).to_excel(xl,  sheet_name="PR_detail", index=False)
print("Saved Excel 1:", xl1)
inferential

Saved Excel 1: D:\Headway\Tables\01_inferential_stats.xlsx


,Variable,Scale,Test,Symbol,ES_Overall,Mag_Overall,p_Overall,ES_BTW,Mag_BTW,p_BTW,ES_PR,Mag_PR,p_PR
0,V_Target,bin,Mann-Whitney U,r,0.4745,medium,<0.001,NaN,,n/a,NaN,,n/a
1,V_Leading_Class,multi,Kruskal-Wallis,ε²,0.0248,small,<0.001,0.0706,medium,<0.001,0.0130,small,0.562
2,Pair,multi,Kruskal-Wallis,ε²,0.1775,large,<0.001,0.0706,medium,<0.001,0.0130,small,0.562
3,Target_Speed_km/hr,cont,Spearman,ρ,-0.3901,medium,<0.001,-0.3126,medium,<0.001,-0.2505,small,0.0011
4,Leading_Speed_km/hr,cont,Spearman,ρ,-0.0194,negligible,0.5624,-0.0107,negligible,0.783,0.0724,negligible,0.4738
5,Speed_Difference,cont,Spearman,ρ,-0.3404,medium,<0.001,-0.3126,medium,<0.001,-0.1917,small,0.0161
6,Off_centeredness,bin,Mann-Whitney U,r,-0.1229,small,0.0033,-0.1347,small,0.0067,-0.0808,negligible,0.4738
7,Occupancy,bin,Mann-Whitney U,r,-0.1697,small,<0.001,-0.0545,negligible,0.4258,-0.1854,small,0.0562
8,Flow_pcu/hr,cont,Spearman,ρ,0.1764,small,<0.001,0.1974,small,<0.001,0.1669,small,0.0343
9,Site,bin,Mann-Whitney U,r,0.1665,small,<0.001,0.2431,small,<0.001,0.0530,negligible,0.562


In [5]:
# --- Cell 5: STEP 2 (NEW) - VIF multicollinearity check across covariates -> Excel 2 ---
def compute_vif(frame, cols):
    X = frame[cols].apply(pd.to_numeric, errors="coerce").dropna()
    R = np.corrcoef(X.values.T)
    try:    iR = np.linalg.inv(R)
    except np.linalg.LinAlgError: iR = np.linalg.pinv(R)
    return {c: round(float(iR[i, i]), 3) for i, c in enumerate(cols)}

vif_cols = [v["col"] for v in COVARIATES.values()]
name_of  = {v["col"]: k for k, v in COVARIATES.items()}
vif_global = compute_vif(df, vif_cols)
vif_tbl = pd.DataFrame([dict(Covariate=name_of[c], VIF=vif_global[c],
                             flag=("high (>10)" if vif_global[c] > 10 else
                                   "moderate (>5)" if vif_global[c] > VIF_THRESH else "ok"))
                        for c in vif_cols]).sort_values("VIF", ascending=False).reset_index(drop=True)

# correlation matrix among continuous covariates (context for the VIF)
cont_cols = [v["col"] for v in COVARIATES.values() if v["type"] == "cont"]
corr_cont = df[cont_cols].corr(method="spearman").round(3)
corr_cont.index   = [name_of[c] for c in corr_cont.index]
corr_cont.columns = [name_of[c] for c in corr_cont.columns]

xl2 = os.path.join(TABLES, "02_vif_multicollinearity.xlsx")
with pd.ExcelWriter(xl2, engine="openpyxl") as xl:
    vif_tbl.to_excel(xl,   sheet_name="VIF_global", index=False)
    corr_cont.to_excel(xl, sheet_name="Corr_continuous")
print("Saved Excel 2:", xl2)
print("Note: speed & speed_diff low VIF -> may enter together; high-VIF pairs are near-substitutes.")
vif_tbl

Saved Excel 2: D:\Headway\Tables\02_vif_multicollinearity.xlsx
Note: speed & speed_diff low VIF -> may enter together; high-VIF pairs are near-substitutes.


,Covariate,VIF,flag
0,site,14.512,high (>10)
1,flow,13.378,high (>10)
2,speed,1.928,ok
3,speed_diff,1.295,ok
4,occupancy,1.067,ok
5,off_cen,1.019,ok


In [6]:
# --- Cell 6: Marginal distribution fit helper (Step 3) ---
def fit_marginal(name, data):
    dist = getattr(stats, name)
    try:
        params = dist.fit(data)
        ll = np.sum(dist.logpdf(data, *params))
        if not np.isfinite(ll): return None
        k, n = len(params), len(data)
        ks, ksp = stats.kstest(data, name, args=params)
        shapes = (dist.shapes.split(",") if dist.shapes else [])
        labels = [s.strip() for s in shapes] + ["loc", "scale"]
        pstr = ", ".join(f"{l}={v:.4f}" for l, v in zip(labels, params))
        return dict(Distribution=name, k=k, LogLik=ll, AIC=2*k-2*ll, BIC=k*np.log(n)-2*ll,
                    KS=ks, KS_p=ksp, Params=pstr)
    except Exception:
        return None

In [7]:
# --- Cell 7: STEP 3 - Fit all candidates per pair (report all) -> Excel 3 ---
all_rows, best_rows, pair_best = [], [], {}
for pair, g in df.groupby("Pair"):
    t = g[OUTCOME].dropna().values; n = len(t)
    fits = [f for f in (fit_marginal(nm, t) for nm in CANDIDATE_DISTS) if f]
    fits.sort(key=lambda d: d["AIC"])
    for rank, f in enumerate(fits, 1):
        all_rows.append(dict(Pair=pair, N=n, Rank=rank, Distribution=f["Distribution"],
                             AIC=round(f["AIC"],2), BIC=round(f["BIC"],2),
                             KS=round(f["KS"],4), KS_p=round(f["KS_p"],4), Params=f["Params"]))
    b = fits[0]; pair_best[pair] = b["Distribution"]
    best_rows.append(dict(Pair=pair, N=n, Best_Distribution=b["Distribution"],
                          extreme_value_flag="Yes" if b["Distribution"] in FLAG_DISTS else "No",
                          AIC=round(b["AIC"],2), KS_p=round(b["KS_p"],4),
                          fit_flag="ok" if n>=MIN_N_FIT else ("small" if n>=15 else "TOO SMALL"),
                          Params=b["Params"]))
all_fits = pd.DataFrame(all_rows)
best_per_pair_dist = pd.DataFrame(best_rows).sort_values("N", ascending=False).reset_index(drop=True)

xl3 = os.path.join(TABLES, "03_distribution_fits.xlsx")
with pd.ExcelWriter(xl3, engine="openpyxl") as xl:
    best_per_pair_dist.to_excel(xl, sheet_name="Best_per_pair", index=False)
    all_fits.to_excel(xl,           sheet_name="All_fits", index=False)
print("Saved Excel 3:", xl3)
best_per_pair_dist

Saved Excel 3: D:\Headway\Tables\03_distribution_fits.xlsx


,Pair,N,Best_Distribution,extreme_value_flag,AIC,KS_p,fit_flag,Params
0,BTW_following_4W,250,weibull_min,No,622.44,0.9187,ok,"c=2.4049, loc=0.3652, scale=2.1645"
1,BTW_following_MT_3W,186,weibull_min,No,440.28,0.9001,ok,"c=1.6347, loc=0.4816, scale=1.5418"
2,BTW_following_NMT_3W,119,gengamma,No,293.42,0.8286,ok,"a=0.3686, c=3.0339, loc=0.5301, scale=2.5862"
3,PR_following_MT_3W,104,genextreme,Yes,280.33,0.6508,ok,"c=0.3302, loc=2.5547, scale=0.9291"
4,BTW_following_MT_2W,73,gengamma,No,162.79,0.8259,ok,"a=0.4376, c=2.2363, loc=0.6000, scale=2.1461"
5,PR_following_NMT_3W,49,genextreme,Yes,142.14,0.6040,ok,"c=0.3738, loc=2.4029, scale=1.0178"
6,PR_following_4W,43,gengamma,No,129.53,0.5656,ok,"a=0.1499, c=8.7833, loc=0.8197, scale=3.8736"
7,BTW_following_NMT_2W,41,gengamma,No,70.29,0.0893,ok,"a=0.3735, c=2.0770, loc=0.5667, scale=2.0776"
8,PR_following_MT_2W,23,gengamma,No,60.70,0.5246,small,"a=0.1402, c=5.9398, loc=1.3333, scale=3.7174"
9,PR_following_NMT_2W,10,gengamma,No,16.44,0.2439,TOO SMALL,"a=0.2616, c=2.3901, loc=0.6000, scale=4.0779"


In [8]:
# --- Cell 8: STEP 4 - Drop low-sample pairs (EDIT THIS LIST IF NEEDED) ---
DROP_PAIRS = ["PR_following_MT_2W", "PR_following_NMT_2W"]      # keeps 8 pairs
KEPT_PAIRS = (best_per_pair_dist[~best_per_pair_dist["Pair"].isin(DROP_PAIRS)]
              .sort_values("N", ascending=False)["Pair"].tolist())
dfk = df[df["Pair"].isin(KEPT_PAIRS)].copy()
print(f"Dropped: {DROP_PAIRS}")
print(f"Kept ({len(KEPT_PAIRS)}):")
for p in KEPT_PAIRS:
    print(f"   {p:24s} n={int((df['Pair']==p).sum()):4d}  best_dist={pair_best[p]}")

Dropped: ['PR_following_MT_2W', 'PR_following_NMT_2W']
Kept (8):
   BTW_following_4W         n= 250  best_dist=weibull_min
   BTW_following_MT_3W      n= 186  best_dist=weibull_min
   BTW_following_NMT_3W     n= 119  best_dist=gengamma
   PR_following_MT_3W       n= 104  best_dist=genextreme
   BTW_following_MT_2W      n=  73  best_dist=gengamma
   PR_following_NMT_3W      n=  49  best_dist=genextreme
   PR_following_4W          n=  43  best_dist=gengamma
   BTW_following_NMT_2W     n=  41  best_dist=gengamma


In [9]:
# --- Cell 9: Generic AFT machinery (scale ~ covariate, shape shared, loc=0) ---
def robust_min(fn, x0, args):
    best = None
    for m, o in (("Nelder-Mead", {"maxiter":20000,"xatol":1e-8,"fatol":1e-8}),
                 ("Powell",       {"maxiter":20000})):
        r = optimize.minimize(fn, x0, args=args, method=m, options=o)
        if best is None or r.fun < best.fun: best = r
    return best
def _nll_null(p, t, dist, ns):
    lp = dist.logpdf(t, *p[:ns], loc=0, scale=np.exp(p[ns]))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12
def _nll_cov(p, t, z, dist, ns):
    lp = dist.logpdf(t, *p[:ns], loc=0, scale=np.exp(p[ns] + p[ns+1]*z))
    return -lp.sum() if np.all(np.isfinite(lp)) else 1e12
def aft_null(dist_name, t):
    dist = getattr(stats, dist_name); ns = dist.numargs; init = dist.fit(t, floc=0)
    r = robust_min(_nll_null, list(init[:ns]) + [np.log(init[-1])], (t, dist, ns))
    return r, -r.fun, ns
def screen_pair(dist_name, t, g, covariates):
    r0, ll0, ns = aft_null(dist_name, t); out = []
    for name, meta in covariates.items():
        raw = pd.to_numeric(g[meta["col"]], errors="coerce").values.astype(float)
        if meta["type"] == "cont":
            z = raw - np.nanmean(raw); ok = np.nanstd(raw) > 1e-9
        else:
            z = raw; ok = (len(np.unique(raw))==2 and min((raw==0).sum(),(raw==1).sum())>=MIN_BIN_GROUP)
        if not ok:
            out.append(dict(Covariate=name, Effect="-- low variation --", dAIC=np.nan,
                            LR_p=np.nan, improves_strict="n/a")); continue
        r1 = robust_min(_nll_cov, list(r0.x) + [0.0], (t, z, getattr(stats, dist_name), ns))
        b1 = r1.x[ns+1]; LR = 2*(-r1.fun - ll0); p = stats.chi2.sf(LR, 1); dAIC = LR - 2
        eff = (f"{(np.exp(b1*meta['unit'])-1)*100:+.2f}%  {meta['label']}" if meta["type"]=="cont"
               else f"{(np.exp(b1)-1)*100:+.2f}%  ({meta['label']})")
        out.append(dict(Covariate=name, Type=meta["type"], Effect=eff, coef_b1=round(b1,6),
                        LR_chi2=round(LR,2), LR_p="<0.001" if p<1e-3 else round(p,4),
                        dAIC=round(dAIC,2),
                        improves="Yes" if p<ALPHA else "No",
                        improves_strict="Yes" if (p<ALPHA and dAIC>=DAIC_MIN) else "No",
                        converged=bool(r1.success)))
    return out

In [10]:
# --- Cell 10: STEP 5 - Covariate screen per pair under its OWN best distribution -> Excel 4 ---
rows = []
for pair in KEPT_PAIRS:
    g = dfk[dfk["Pair"] == pair]; t = g[OUTCOME].values; n = len(t)
    dname = pair_best[pair]
    for rec in screen_pair(dname, t, g, COVARIATES):
        rec.update(Pair=pair, N=n, Distribution=dname,
                   ev_flag="Yes" if dname in FLAG_DISTS else "No")
        rows.append(rec)
screen = pd.DataFrame(rows)[
    ["Pair","N","Distribution","ev_flag","Covariate","Type","Effect","coef_b1",
     "LR_chi2","LR_p","dAIC","improves","improves_strict","converged"]]

order_cov = list(COVARIATES.keys())
num = screen.copy(); num["d"] = pd.to_numeric(num["dAIC"], errors="coerce")
dAIC_matrix = num.pivot(index="Pair", columns="Covariate", values="d").reindex(index=KEPT_PAIRS, columns=order_cov).round(2)

xl4 = os.path.join(TABLES, "04_covariate_screen_ownbest.xlsx")
with pd.ExcelWriter(xl4, engine="openpyxl") as xl:
    screen.to_excel(xl,       sheet_name="Screening_long", index=False)
    dAIC_matrix.to_excel(xl,  sheet_name="dAIC_matrix")
print("Saved Excel 4:", xl4)
print("EV-distribution pairs (interpret covariates with care):",
      [p for p in KEPT_PAIRS if pair_best[p] in FLAG_DISTS])
dAIC_matrix

Saved Excel 4: D:\Headway\Tables\04_covariate_screen_ownbest.xlsx
EV-distribution pairs (interpret covariates with care): ['PR_following_MT_3W', 'PR_following_NMT_3W']


Covariate,speed,speed_diff,flow,off_cen,occupancy,site
Pair,,,,,,
BTW_following_4W,26.78,29.76,4.25,1.83,-0.54,4.28
BTW_following_MT_3W,34.26,26.81,14.13,-1.41,2.58,13.11
BTW_following_NMT_3W,3.98,-1.57,3.15,-0.78,-1.39,0.77
PR_following_MT_3W,11.68,0.97,5.12,11.31,10.87,10.09
BTW_following_MT_2W,3.25,5.61,0.36,-0.44,-2.00,0.20
PR_following_NMT_3W,-0.98,-1.42,17.89,-1.91,14.65,16.74
PR_following_4W,-2.00,-1.94,-1.59,-0.37,-1.46,-1.31
BTW_following_NMT_2W,-0.56,7.22,-1.66,-2.00,-1.95,-1.69


In [11]:
# --- Cell 11: STEP 6 - Finalize covariate set (VIF-driven de-correlation) -> Excel 5 ---
def vif_ok(frame, cols):
    """Max VIF of the given covariate columns within this pair's data."""
    if len(cols) < 2: return True, 1.0
    X = frame[cols].apply(pd.to_numeric, errors="coerce").dropna()
    if X.shape[0] < len(cols) + 2 or X.nunique().min() < 2: return True, np.nan
    R = np.corrcoef(X.values.T)
    try:    iR = np.linalg.inv(R)
    except np.linalg.LinAlgError: iR = np.linalg.pinv(R)
    mv = float(np.max(np.diag(iR)))
    return (mv <= VIF_THRESH), round(mv, 2)

num_scr = screen.copy(); num_scr["d"] = pd.to_numeric(num_scr["dAIC"], errors="coerce")
final = []
for pair in KEPT_PAIRS:
    g = dfk[dfk["Pair"] == pair]
    passing = (num_scr[(num_scr["Pair"]==pair) & (num_scr["improves_strict"]=="Yes")]
               .sort_values("d", ascending=False))
    cand = list(passing["Covariate"])
    selected, sel_cols = [], []
    for cov in cand:                                   # greedy add while VIF stays acceptable
        col = COVARIATES[cov]["col"]
        ok, mv = vif_ok(g, sel_cols + [col])
        if ok:
            selected.append(cov); sel_cols.append(col)
    ok_final, maxvif = vif_ok(g, sel_cols)
    dropped = [c for c in cand if c not in selected]
    final.append(dict(Pair=pair, N=int((dfk["Pair"]==pair).sum()),
                      Distribution=pair_best[pair],
                      ev_flag="Yes" if pair_best[pair] in FLAG_DISTS else "No",
                      All_passing=", ".join(cand) if cand else "none",
                      Final_set=", ".join(selected) if selected else "none (plain fit)",
                      dropped_for_collinearity=", ".join(dropped) if dropped else "-",
                      max_VIF=maxvif))
final_sets = pd.DataFrame(final)

xl5 = os.path.join(TABLES, "05_final_covariate_sets.xlsx")
with pd.ExcelWriter(xl5, engine="openpyxl") as xl:
    final_sets.to_excel(xl, sheet_name="Final_model_spec", index=False)
print("Saved Excel 5:", xl5)
final_sets

Saved Excel 5: D:\Headway\Tables\05_final_covariate_sets.xlsx


,Pair,N,Distribution,ev_flag,All_passing,Final_set,dropped_for_collinearity,max_VIF
0,BTW_following_4W,250,weibull_min,No,"speed_diff, speed, site, flow","speed_diff, speed, site",flow,2.10
1,BTW_following_MT_3W,186,weibull_min,No,"speed, speed_diff, flow, site, occupancy","speed, speed_diff, flow, occupancy",site,2.10
2,BTW_following_NMT_3W,119,gengamma,No,"speed, flow","speed, flow",-,1.12
3,PR_following_MT_3W,104,genextreme,Yes,"speed, off_cen, occupancy, site, flow","speed, off_cen, occupancy, site",flow,1.45
4,BTW_following_MT_2W,73,gengamma,No,"speed_diff, speed","speed_diff, speed",-,1.00
5,PR_following_NMT_3W,49,genextreme,Yes,"flow, site, occupancy","flow, occupancy",site,1.37
6,PR_following_4W,43,gengamma,No,none,none (plain fit),-,1.00
7,BTW_following_NMT_2W,41,gengamma,No,speed_diff,speed_diff,-,1.00
